# Practice Session 07: Network models


<font size="+2" color="blue">Additional results: fitting of power-law</font>

Author: <font color="blue">Luca Franceschi</font>

E-mail: <font color="blue">luca.franceschi01@estudiant.upf.edu</font>

Date: <font color="blue">Due Nov. 3rd, 20:30</font>

# 1. Random (ER) graph generator

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import itertools

from collections import OrderedDict

In [ ]:
# Leave as-is

def flip_coin(p):
    if np.random.random() < p:
        return True
    else:
        return False

## 1.1. Generate ER graph

In [ ]:
def generate_random_graph(N, p):
    graph = nx.Graph() # Create empty graph
    graph.add_nodes_from([i for i in range(N)]) # Add nodes numbered 0-(N-1) using list comprehension
    for u, v in itertools.combinations(graph.nodes, 2): # Iterate over all pairs of nodes in the graph
        if flip_coin(p):
            graph.add_edge(u, v) # Assign edge with probability p
    return graph

In [ ]:
N = 500
trials = 300
p = 0.5

expected_edges = N*(N-1)/2 * p
observed_edges = np.zeros(trials)

for i in range(trials):
    g = generate_random_graph(N, p)
    observed_edges[i] = g.number_of_edges()

In [ ]:
# Leave as-is

plt.xlabel("Run")
plt.ylabel("Number of edges")
plt.plot(range(trials), observed_edges, label="Observed")
plt.plot(range(trials), [expected_edges] * trials, label="Expected")
plt.legend()
plt.show()

In [ ]:
def generate_random_graph_avg_degree(N, kavg):
    graph = nx.Graph()
    graph.add_nodes_from([i for i in range(N)])  # Add nodes 0 to N-1

    p = kavg/(N-1) # Calculate p from kavg and number of nodes-1

    for u, v in itertools.combinations(graph.nodes, 2):
        if flip_coin(p): # Add edge with probability p
            graph.add_edge(u, v)
    return graph

In [ ]:
def graph_average_degree(graph):
    return 2* graph.number_of_edges() / graph.number_of_nodes()

target_average_degrees = np.arange(0, 3.1, 0.1)
observed_average_degrees = np.zeros(len(target_average_degrees))

for i in range(len(target_average_degrees)):
    g = generate_random_graph_avg_degree(N, target_average_degrees[i])
    observed_average_degrees[i] = graph_average_degree(g)

In [ ]:
# Leave as-is

plt.xlabel("Run")
plt.ylabel("Number of edges")
plt.plot(range(len(target_average_degrees)), observed_average_degrees, label="Observed")
plt.plot(range(len(target_average_degrees)), target_average_degrees, label="Target")
plt.legend()
plt.show()

## 1.2. Measure connectivity


In [ ]:
def is_connected(g):
    for u, v in itertools.combinations(g.nodes, 2): # Iterate over all pair of nodes in the graph
        if not nx.has_path(g, u, v): # If does not exist path, the graph is not connected
            return False
    return True # If after searching all nodes all paths exist then is connected

In [ ]:
# Leave as-is

def size_largest_cc(G):
    
    # Obtain the list of connected components of the graph sorted from largest to smallest
    Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
    
    # Selects the first connected component in that list
    G0 = G.subgraph(Gcc[0])
    
    # Returns its number of nodes
    return G0.number_of_nodes()

In [ ]:
N = 500
target_average_degrees = np.arange(0.3, 3.0, 0.05)

largest_cc_sizes = []
average_degrees = []

for target_average_degree in target_average_degrees:
    
    # Generate graph
    g = generate_random_graph_avg_degree(N, target_average_degree)
    assert g.number_of_nodes() == N, "Wrong number of nodes"
    
    # Obtain observed average degree
    average_degree = graph_average_degree(g)
    average_degrees.append(average_degree)
    
    # Obtain size of largest connected component as a fraction of the total number of nodes
    largest_cc_size = size_largest_cc(g)/N
    largest_cc_sizes.append(largest_cc_size)

In [ ]:
plt.figure(figsize=(12,6))
plt.scatter(average_degrees, largest_cc_sizes)
plt.title("Scatter plot: size of largest connected component")
plt.xlabel("Observed average degree <k>")
plt.ylabel("Size of largest connected component")
plt.show()

In [ ]:
# LEAVE AS-IS

plt.figure(figsize=(12,6))
plt.plot(average_degrees, largest_cc_sizes)
plt.title("Size of largest connected component")
plt.xlabel("Observed average degree <k>")
plt.ylabel("Size of largest connected component")
plt.show()

1. The theory says that the largest connected component size increases substantially when the average degree is just above 1.

2. In this particular case the theory seems to be right, shortly after the critical point of average_degree=1 the size of the largest connected component increases from about 0.2 to almost 0.8, which is a very significant slope.

3. The resulting curve is not a function because there could be the case that we can find two graphs with same observed average degree but different size of largest connected component. Therefore this could be seen as a non-bijective (or one-to-many) function.

## 1.3. Visualize graphs and plot degree distributions

In [ ]:
def plot_graph(g, node_size=50):
    plt.figure(figsize=(12,6))
    plt.axis('off')
    pos=nx.spring_layout(g)
    nx.draw_networkx(g, pos, with_labels=False, node_size=node_size, node_color='lightgreen')

In [ ]:
p = 0.1
graphs = []
total = 0
while len(graphs) < 3:
    graph = generate_random_graph(100, p)
    if is_connected(graph):
        plot_graph(graph)
        graphs.append(graph)
    total += 1
print('Total iterations to find a connected graph: ', total)

In [ ]:
def print_er_statistics(g, p):
    obs_kavg = graph_average_degree(g)
    exp_kavg = p * (g.number_of_nodes()-1)
    print(f'Expected <k>: {exp_kavg:.2f}\t Observed <k>: {obs_kavg:.2f}')

In [ ]:
for g in graphs:
    print_er_statistics(g, p)

In [ ]:
# Leave as-is or modify if you want

def plot_degree_distribution(g):
    degree_dict = dict(g.degree())
    degree_ordered = OrderedDict(sorted(degree_dict.items(), key=lambda x: x[1], reverse=True))
    degree_sequence = list(degree_ordered.values())
    prob, bin_edges = np.histogram(degree_sequence, bins=range(1,np.max(degree_sequence)+2), density=True)
    
    plt.figure(figsize=(12,6))
    plt.loglog(bin_edges[:-1], prob, 'o-')
    plt.title("Probability density function")
    plt.xlabel("degree")
    plt.ylabel("probability")
    plt.autoscale(enable=True, axis='both')
    plt.show()

In [ ]:
ps = [0.0015, 0.0025, 0.005, 0.01, 0.02]

for i, n in zip(range(5), range(1500, 3001, 300)):
    graph = generate_random_graph(n, ps[i])
    plot_graph(graph, 10)
    plot_degree_distribution(graph)
    print_er_statistics(graph, ps[i])

In the plots above we can see that as the probability of adding an edge in a random network grows, so does the degrees of the nodes in average. We can see that in the first graph there are a majority of nodes that have degree 0. In the second graph that amount of nodes is reduced to just a few nodes. From there on, all graphs are normally connected and have a relatively high average degree, despite the probability of adding an edge being still quite small (highest is 0.2).

# 2. Preferential attachment (BA) generator

In [ ]:
# LEAVE AS-IS

def select_with_probability(x, m, p):
    return np.random.choice(x, size=m, replace=False, p=p)

In [ ]:
# LEAVE AS-IS

trials = 2000
vector = ['a', 'b', 'c', 'd']
probabilities = [0.60, 0.15, 0.12, 0.13]

all_selected = []
for i in range(trials):
    selected = select_with_probability(vector, 1, probabilities)[0]
    all_selected.append(selected)
    
for i, p in zip(vector, probabilities):
    print("Element {:s} was selected {:d} times, expected {:.0f}".format(i, all_selected.count(i), trials*p) )

In [ ]:
def select_targets(g, m):

    # Check if feasible
    N = g.number_of_nodes()  
    if N < m:
        raise ValueError('Graph has less than m nodes')

    # Compute sum of degree
    sum_degree = 0

    # YOUR CODE HERE: COMPUTE SUM OF DEGREE OF NODES
    for (node, degree) in g.degree():
        sum_degree += degree

    if sum_degree == 0:
        raise ValueError('Graph as no edges')

    # Compute probabilities
    probabilities = []
    for (node, degree) in g.degree():
        # YOUR CODE HERE: COMPUTE PROBABILITY OF SELECTING NODE u
        prob = degree / sum_degree
        # THEN APPEND IT TO probabilities USING probabilities.append(...)
        probabilities.append(prob)
    # Sample
    selected = select_with_probability(g.nodes(), m, probabilities)

    return selected

In [ ]:
def generate_preferential_attachment_graph(N, m0, m):
    if m > m0:
        raise ValueError('m not <= m0') # Check m<=m0
    
    graph = nx.Graph()
    graph.add_nodes_from([i for i in range(m0)]) # Add nodes from 0 to m0-1

    for i in range(m0-1):
        graph.add_edge(i, i+1) # Add edges to create a cycle of nodes 0 to m-1
    graph.add_edge(m0-1, 0) # Last node with first

    for i in range(m0, N):
        selected = select_targets(graph, m) # Select before adding node 
        graph.add_node(i)
        selected = [(i, selected[j]) for j in range(len(selected))] # Edges that will be added are i to all selected nodes
        graph.add_edges_from(selected)

    return graph

In [ ]:
def plot_degree_distribution_BA(g, alpha): # Used for the extra
    degree_dict = dict(g.degree())
    degree_ordered = OrderedDict(sorted(degree_dict.items(), key=lambda x: x[1], reverse=True))
    degree_sequence = list(degree_ordered.values())
    prob, bin_edges = np.histogram(degree_sequence, bins=range(1,np.max(degree_sequence)+2), density=True)

    x = np.linspace(degree_sequence[-1], degree_sequence[0], degree_sequence[0]-degree_sequence[-1]) # xmin is 1
    power_law = np.power(x, -alpha)

    plt.figure(figsize=(12,6))
    plt.loglog(bin_edges[:-1], prob, 'o-')
    plt.loglog(x, power_law)
    plt.title("Probability density function")
    plt.xlabel("degree")
    plt.ylabel("probability")
    plt.autoscale(enable=True, axis='both')
    plt.show()

In [ ]:
alpha = 0.45

graph = generate_preferential_attachment_graph(1000, 2, 1)
plot_graph(graph, 10)
plot_degree_distribution_BA(graph, 1/alpha)

In this graph we can see that the distribution of the degrees follows a power law. That is because on generation the nodes that have a high degree are the ones more likely to have another edge added. This ends up in a graph that has a noticeable difference between 'central' nodes and 'peripheral' nodes because there are much more nodes with low degree, and a very few nodes with an extremely high degree. We can see that because this graph in particular is quite sparse due to the parameters of the generator function.

We can also see that if we fit a power law in the plot (in orange), the probability of having a certain degree follow a power law distribution, which appears to be lineal because the scale is logarithmic.

In [ ]:
graph = generate_preferential_attachment_graph(2500, 7, 5)
plot_graph(graph, 10)
plot_degree_distribution_BA(graph, 1/alpha)

In this graph we are not able to have such a good view since it is very dense, however we can see in the probability density function that it follows the power law. As before, since when the graph is generated there is a higher chance of selecting a node with higher degree than another that does not have such high degree, the 'central' nodes end up being much more connected between them, hence the difference between them and other 'peripheral' nodes is much more noticeable.

We can also fit a power law distribution in the pdf plot and it fits quite nicely.

In [ ]:
N = 1000

graph = generate_preferential_attachment_graph(N, 3, 3)
nodes = np.linspace(0, N, N)
degrees = np.zeros(N)

for (node, degree) in graph.degree():
    degrees[node] = degree

x = np.linspace(1, N, N-1) # xmin is 1
power_law = np.power(x, -alpha)
power_law *= degrees.max() # Scale it

plt.figure(figsize=(12, 6))
plt.title("Degree per node scatter plot")
plt.xlabel("Node id")
plt.ylabel("Degree")
plt.scatter(nodes, degrees)
plt.plot(power_law, color='orange')
plt.plot()
plt.plot()
plt.show()

We can see that nodes with lower id have much higher degree than others. That is probably due to the fact that, when generating a BA graph, we start adding some links between the first m0 nodes, hence they start with higher probability than others. That leads to a 'snowball' effect that ends up leading those nodes with a higher probability every iteration.
As a resut, The nodes with lower id end up having a higher degree due to the process of generation of BA graphs.

In particular, the degrees versus node id distribution, as in previous plots, follows a power law distribution. In this case, we can interpret it as follows: there are a very few nodes (the first ones) which have high degree, while the vast majority of nodes have a degree of a few nodes (around 5 to 10 nodes)

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>